In [ ]:
import numpy as np
import matplotlib.pyplot as plt


We would like to optimise the function below

$$
f(x_1,x_2) = -\exp\big(-(x_1 x_2 - 1.5)^2-(x_2-1.5)^2\big) - 0.6\,\exp\!\left(-\frac{(x_1-1)^2+(x_2-2.2)^2}{0.15}\right),
$$

using gradient descent algorithms. This function has two basins: the shallow one near $(2, 1)$ and a deeper, narrower one near $(0.9, 2.0)$, so the path taken by the optimiser (and the step size it uses) actually matters.

The starting point will be placed at $x_0 = [2.5, 0.5]$.

The function is defined below and plotted.

In [ ]:
def f(x):
    """x is a length-2 array-like [x1, x2]. Returns a scalar."""
    x1, x2 = x[0], x[1]
    return (-np.exp(-(x1*x2 - 1.5)**2 - (x2 - 1.5)**2)
            - 0.6*np.exp(-((x1 - 1)**2 + (x2 - 2.2)**2) / 0.15))


In [ ]:
# plot the function
nx = 100
ny = 100
x = np.linspace(0, 3, nx+1)
y = np.linspace(0, 3, ny+1)
X, Y = np.meshgrid(x, y, indexing='ij')

F = f([X, Y])

plt.contour(X, Y, F, levels=30)
plt.axis('image')
plt.show()


Previously the gradient was obtained symbolically with `sympy`. In practice, for most real problems you don't have (or don't want to derive) a closed-form gradient, so instead we estimate it numerically at whatever point we need it, using **finite differences**.

For a small step $h$, the central difference approximation of a partial derivative is

$$
\frac{\partial f}{\partial x_i}(x) \approx \frac{f(x + h e_i) - f(x - h e_i)}{2h}.
$$

**>>> Task:** complete the function below so that it returns the gradient of `f` at `x`, estimated by central differences.

In [ ]:
def numerical_gradient(f, x, h=1e-6):
    """Estimate the gradient of f at x using central finite differences.

    f : callable, f(x) -> scalar
    x : array-like, point at which to estimate the gradient
    h : step size used in the finite-difference approximation
    """
    x = np.array(x, dtype=float)
    grad = np.zeros_like(x)

    for i in range(len(x)):
        #>>> build x_plus and x_minus, each equal to x but shifted by +-h on entry i

        #>>> grad[i] = ... central difference formula
        pass

    return grad


You can sanity-check your implementation against the (known) analytical gradient at a test point before moving on, e.g. by comparing `numerical_gradient(f, [2.5, 0.5])` to a hand-derived value.

## (a) Steepest descent (fixed step size)

Initialise the problem

In [ ]:
x0 = np.array([2.5, 0.5])       # First initial guess
x = x0.copy()
eps = 1e-3                       # >>> Precision (on the gradient norm, not the step length)
step_size = 0.1                  # fixed step length along the descent direction

# variables defined for plotting
stepDiffTab = np.array([])
stepDiff = 1                     # Gradient norm, init at 1 for the loop
stepCounter = 0                  # Number of steps


The optimisation loop

In [ ]:
trajectory_sd = x0.reshape(2, 1).copy()   # Trajectory of x

while (stepDiff > eps) and (stepCounter < 100):

    # Calculating dk
    g =                              #>>> estimate the gradient at x using numerical_gradient
    d =                              #>>> normalise the gradient to get a unit descent direction

    x =                               #>>> update x: take a step of length step_size along d
    stepDiff = np.linalg.norm(g)      # stop once the gradient is (near) zero

    # Keeping x values
    trajectory_sd = np.hstack((trajectory_sd, x.reshape(2, 1)))
    # Keeping the gradient norm
    stepDiffTab = np.append(stepDiffTab, stepDiff)
    # printing the value of x at each step
    print('x = ', x)
    stepCounter += 1


Plot the evolution of the solution

In [ ]:
print('minimum of f : x = ', x)

plt.contour(X, Y, F, levels=30)
plt.plot(trajectory_sd[0], trajectory_sd[1], '-o', markersize=3)
plt.axis('image')
plt.show()


## (b) Gradient descent with line search

Instead of taking a fixed-length step along the descent direction $d_k$, we now choose, at every iteration, the step length $\alpha_k$ that (approximately) minimises $f(x_k + \alpha d_k)$ along that direction.

**>>> Task:** complete `line_search` below. It should scan `alpha` over `np.linspace(0, alpha_max, n)`, evaluate $f(x + \alpha d)$ for each value, and return the $\alpha$ that gives the smallest value (a brute-force / grid line search).

In [ ]:
def line_search(f, x, d, alpha_max=1.0, n=100):
    """Brute-force grid line search: returns the alpha in [0, alpha_max]
    (sampled at n points) that minimises f(x + alpha*d).
    """
    alphas =                           #>>> build the grid of candidate step sizes
    values =                           #>>> evaluate f(x + alpha*d) for each candidate

    #>>> return the alpha that minimises f


Initialise the problem

In [ ]:
x = x0.copy()
eps = 1e-3                       # Precision (on the gradient norm)
stepDiffTab = np.array([])
stepDiff = 1                     # Gradient norm, init at 1 for the loop
stepCounter = 0


Optimisation loop

In [ ]:
trajectory_ls = x0.reshape(2, 1).copy()   # Trajectory of x
alpha_max = 1.0
nb_evals_ls = 0                            # count function evaluations used by the line search

while (stepDiff > eps) and (stepCounter < 100):

    # Calculating dk
    g =                                #>>> estimate the gradient at x
    d =                                #>>> normalise it

    # Line search
    alpha =                            #>>> call line_search to find the optimal step size
    nb_evals_ls += 100                 # line_search evaluates f at n=100 points each call

    x =                                #>>> update x using alpha and d
    stepDiff = np.linalg.norm(g)

    # Keeping x values
    trajectory_ls = np.hstack((trajectory_ls, x.reshape(2, 1)))
    stepDiffTab = np.append(stepDiffTab, stepDiff)

    # printing the value of x at each step
    print('x = ', x)
    stepCounter += 1

print('grid line search used about', nb_evals_ls, 'evaluations of f in the line search alone')


In [ ]:
print('minimum of f : x = ', x)

plt.contour(X, Y, F, levels=30)
plt.plot(trajectory_sd[0], trajectory_sd[1], '-o', markersize=3, label='fixed step')
plt.plot(trajectory_ls[0], trajectory_ls[1], '-o', markersize=3, label='grid line search')
plt.axis('image')
plt.legend()
plt.show()


## (c) Implementing a bracketing line search yourselves (golden-section search)

The grid line search above is simple, but it wastes a lot of function evaluations sampling points that are clearly not going to be the minimiser — every call re-evaluates `f` at `n` points from scratch. A **bracketing method** narrows down an interval $[a,b]$ known to contain the minimiser, shrinking it step by step, and can reuse evaluations from one step to the next.

This is **not something covered in the lectures**, so the next few cells build up **golden-section search** in stages, with a task at each stage. By the end you will have your own `golden_section_line_search` to compare against your grid `line_search` from part (b).

We are looking for the $\alpha \in [0, \alpha_{max}]$ minimising $\phi(\alpha) = f(x + \alpha d)$. We assume $\phi$ is **unimodal** on this interval (a single minimum, decreasing then increasing) — reasonable here since $d$ is a descent direction and $\alpha_{max}$ is small.

### Stage 1 — placing two interior points in the bracket

Given a current bracket $[a, b]$ known to contain the minimiser, the idea is to place **two** interior points $c < e$ inside it, evaluate $\phi$ at both, and use the comparison to shrink the bracket (see Stage 2).

Where exactly should $c$ and $e$ go? If we always placed them at, say, the two tercile points, we would need **two brand-new evaluations of $\phi$ every single iteration**. Golden-section search instead places them at

$$
c = b - \gamma (b - a), \qquad e = a + \gamma (b - a), \qquad \gamma = \frac{\sqrt5 - 1}{2} \approx 0.618 \ \text{(the golden ratio)},
$$

so that $c$ and $e$ sit symmetrically inside $[a,b]$. The special property of $\gamma$ is that **one of $\{c, e\}$ from the current step lands exactly on the corresponding point of the new, shrunk bracket** — so only one new evaluation is needed per iteration instead of two (you'll use this in Stage 3).

**>>> Task:** write a function `golden_points(a, b)` that returns `c, e` using the formulas above.

In [ ]:
GR = (np.sqrt(5) - 1) / 2   # ~0.618, the golden ratio

def golden_points(a, b):
    """Return the two interior points c < e of the bracket [a, b],
    placed according to the golden ratio.
    """
    #>>> c = ...
    #>>> e = ...
    return c, e


Quick check: for `a, b = 0.0, 1.0` you should get `c \u2248 0.382` and `e \u2248 0.618`. Try it below.

In [ ]:
print(golden_points(0.0, 1.0))


### Stage 2 — the shrink rule

Now evaluate $\phi(c)$ and $\phi(e)$. Because $\phi$ is unimodal:

- if $\phi(c) < \phi(e)$, the minimum cannot lie in $(e, b]$ (the function would have to turn back up and then down again, which unimodality rules out) — so the new bracket is $[a, e]$.
- otherwise, the minimum cannot lie in $[a, c)$ — so the new bracket is $[c, b]$.

**>>> Task:** write `shrink_bracket(a, b, c, e, fc, fe)` that returns the new `(a, b)` bracket implementing this rule (you don't need to worry about reusing evaluations yet — that's Stage 3).

In [ ]:
def shrink_bracket(a, b, c, e, fc, fe):
    """Given bracket [a,b], interior points c<e and their function values fc, fe,
    return the new, smaller (a, b) bracket.
    """
    if fc < fe:
        #>>> new bracket when phi(c) < phi(e)
        pass
    else:
        #>>> new bracket when phi(c) >= phi(e)
        pass

    return a, b


### Stage 3 — putting it together, without wasting evaluations

A naive loop would call `golden_points` and evaluate $\phi$ at **both** new points every iteration — two evaluations per step. But notice: if the new bracket is $[a, e]$ (old $a$, old $e$), then the old $c$ becomes the new $e$! (You can check this algebraically from the formulas in Stage 1, or just verify it numerically below.) Symmetrically, if the new bracket is $[c, b]$, the old $e$ becomes the new $c$.

So at each iteration, only **one** of the two interior points is actually new, and its function value must be computed; the other one (and its value) can simply be recycled from the previous iteration.

**>>> Task:** complete the loop below so that it keeps the bracket, current `c`, `e`, `fc`, `fe` up to date, calling `phi` **only once per iteration**, until `b - a < tol`. Return `(a + b) / 2` as the estimated minimiser.

In [ ]:
def golden_section_line_search(f, x, d, alpha_max=1.0, tol=1e-4):
    """Golden-section search for the alpha in [0, alpha_max] minimising f(x + alpha*d).
    Must call phi (i.e. f) only once per iteration after the initial setup.
    """
    phi = lambda alpha: f(x + alpha * d)

    a, b = 0.0, alpha_max
    c, e = golden_points(a, b)
    fc, fe = phi(c), phi(e)          # the only two evaluations paid up-front

    while (b - a) > tol:

        if fc < fe:
            #>>> the new bracket is [a, e]; the old c becomes the new e (reuse fc as fe)
            #>>> compute the single new point c and its value fc
            pass
        else:
            #>>> the new bracket is [c, b]; the old e becomes the new c (reuse fe as fc)
            #>>> compute the single new point e and its value fe
            pass

        a, b = shrink_bracket(a, b, c, e, fc, fe)

    return (a + b) / 2


You can sanity-check `golden_section_line_search` against your `line_search` from part (b) on a single step, e.g. from `x0` along its steepest-descent direction — the two should return a similar `alpha`.

Initialise the problem

In [ ]:
x = x0.copy()
eps = 1e-3
stepDiff = 1
stepCounter = 0
nb_evals_gs = 0                  # count function evaluations used by the line search

trajectory_gs = x0.reshape(2, 1).copy()


Optimisation loop using your golden-section line search.

**>>> Task:** fill in the gradient descent loop, reusing the same pattern as parts (a) and (b), but with `golden_section_line_search` in place of `line_search`.

In [ ]:
while (stepDiff > eps) and (stepCounter < 100):

    g =                                 #>>> estimate the gradient at x
    d =                                 #>>> normalise it

    alpha =                             #>>> call golden_section_line_search to find the optimal step size

    x =                                 #>>> update x using alpha and d
    stepDiff = np.linalg.norm(g)

    trajectory_gs = np.hstack((trajectory_gs, x.reshape(2, 1)))
    print('x = ', x)
    stepCounter += 1

print('minimum of f : x = ', x, ' after', stepCounter, 'iterations')


To count evaluations used by the golden-section search itself, you can instrument `golden_section_line_search` (e.g. add a counter argument or a global counter) and compare the total to `nb_evals_ls` from part (b). As a shortcut: each call does 2 evaluations up front, then exactly 1 per while-loop iteration — so the total is `2 + (number of while-loop iterations)`, versus a flat `n=100` for the grid search, every single outer iteration.

In [ ]:
plt.contour(X, Y, F, levels=30)
plt.plot(trajectory_sd[0], trajectory_sd[1], '-o', markersize=3, label='fixed step')
plt.plot(trajectory_ls[0], trajectory_ls[1], '-o', markersize=3, label='grid line search')
plt.plot(trajectory_gs[0], trajectory_gs[1], '-o', markersize=3, label='golden-section line search')
plt.axis('image')
plt.legend()
plt.show()


**Discussion:**
- Does your golden-section version converge to the same $x$ as the grid line search? In roughly the same number of *outer* iterations?
- Roughly how many evaluations of $f$ did the golden-section search use in total, compared to the grid search?
- Which method(s) escape the shallow basin near $(2, 1)$ and find the deeper one near $(0.9, 2.0)$? What happens if you start from a different $x_0$, or shrink `alpha_max`?